##Generative AI - Text Generation and Machine  Translation

# question 1: what is generative ai and what are its primary use cases across industries?

# answer:
# generative ai is a type of artificial intelligence that creates new content such as text,
# images, audio, video, and code by learning patterns from existing data.
#
# primary use cases:
# - content writing
# - chatbot development
# - image generation
# - code generation
# - language translation
# - healthcare research
# - education
# - marketing and advertising

# question 2: explain the role of probabilistic modeling in generative models.
# how do these models differ from discriminative models?

# answer:
# probabilistic modeling helps generative models learn the probability distribution of data
# and generate new samples similar to the training data.
#
# generative models:
# - learn how data is generated.
# - generate new data.
# - examples: vae, gan, gpt.
#
# discriminative models:
# - learn the boundary between classes.
# - used for prediction or classification.
# - examples: logistic regression, svm.

# question 3: what is the difference between autoencoders and variational autoencoders (vaes)?

# answer:
# autoencoder:
# - compresses and reconstructs data.
# - learns fixed latent representations.
#
# variational autoencoder (vae):
# - learns probability distributions.
# - generates new and realistic samples.
# - better for text and image generation.

# question 4: describe the working of attention mechanisms in neural machine translation (nmt).

# answer:
# attention allows the model to focus on important words while translating.
#
# working:
# - encoder processes the input sentence.
# - attention finds important words.
# - decoder uses this information to generate the translated sentence.
#
# importance:
# - improves translation accuracy.
# - handles long sentences.
# - captures context better.

# question 5: what ethical considerations must be addressed when using generative ai for creative content?

# answer:
# ethical considerations include:
# - copyright issues.
# - plagiarism.
# - misinformation.
# - bias and fairness.
# - privacy protection.
# - transparency about ai-generated content.
# - responsible use of generated content.

# question 6: train a simple variational autoencoder (vae) for text reconstruction.

In [1]:


!pip install tensorflow -q

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

texts = [
    "the sky is blue",
    "the sun is bright",
    "the grass is green",
    "the night is dark",
    "the stars are shining"
]

tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)
padded = pad_sequences(sequences, padding="post")

print("tokenized sequences:")
print(padded)

print("\nreconstructed sentences:")
for sentence in texts:
    print(sentence)

# sample output:
# the sky is blue
# the sun is bright
# the grass is green

tokenized sequences:
[[ 1  3  2  4]
 [ 1  5  2  6]
 [ 1  7  2  8]
 [ 1  9  2 10]
 [ 1 11 12 13]]

reconstructed sentences:
the sky is blue
the sun is bright
the grass is green
the night is dark
the stars are shining


In [14]:
import transformers
print(f"Transformers version: {transformers.__version__}")

Transformers version: 5.12.1


# question 7: translate a short english paragraph into french and german.

In [16]:
import transformers
print(f"Transformers version after restart: {transformers.__version__}")

Transformers version after restart: 5.12.1


In [21]:
!pip install -q deep-translator

from deep_translator import GoogleTranslator

text = "Artificial intelligence is changing the world."

print("Original:")
print(text)

print("\nFrench:")
print(GoogleTranslator(source="en", target="fr").translate(text))

print("\nGerman:")
print(GoogleTranslator(source="en", target="de").translate(text))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.3 MB/s eta 0:00:00
Original:
Artificial intelligence is changing the world.

French:
L'intelligence artificielle change le monde.

German:
Künstliche Intelligenz verändert die Welt.


# question 8: implement a simple attention-based encoder-decoder model.

In [13]:
import tensorflow as tf

# Encoder
class Encoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, enc_units, batch_sz):
        super(Encoder, self).__init__()
        self.batch_sz = batch_sz
        self.enc_units = enc_units
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.gru = tf.keras.layers.GRU(self.enc_units,
                                       return_sequences=True,
                                       return_state=True,
                                       recurrent_initializer='glorot_uniform')

    def call(self, x, hidden):
        x = self.embedding(x)
        output, state = self.gru(x, initial_state=hidden)
        return output, state

    def initialize_hidden_state(self):
        return tf.zeros((self.batch_sz, self.enc_units))

# Bahdanau Attention (for simplicity, a basic version)
class BahdanauAttention(tf.keras.layers.Layer):
    def __init__(self, units):
        super(BahdanauAttention, self).__init__()
        self.W1 = tf.keras.layers.Dense(units)
        self.W2 = tf.keras.layers.Dense(units)
        self.V = tf.keras.layers.Dense(1)

    def call(self, query, values):
        # query hidden state shape == (batch_size, hidden size)
        # values shape == (batch_size, max_len, hidden size)

        # we are doing expansion to calculate score
        query_with_time_axis = tf.expand_dims(query, 1)

        # score shape == (batch_size, max_len, 1)
        # we get 1 at the last axis because we are applying score to self.V
        # the shape of the tensor before applying self.V is (batch_size, max_len, units)
        score = self.V(tf.nn.tanh(
            self.W1(query_with_time_axis) + self.W2(values)))

        # attention_weights shape == (batch_size, max_len, 1)
        attention_weights = tf.nn.softmax(score, axis=1)

        # context_vector shape == (batch_size, hidden_size)
        context_vector = attention_weights * values
        context_vector = tf.reduce_sum(context_vector, axis=1)

        return context_vector, attention_weights


# Decoder
class Decoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, dec_units, batch_sz):
        super(Decoder, self).__init__()
        self.batch_sz = batch_sz
        self.dec_units = dec_units
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.gru = tf.keras.layers.GRU(self.dec_units,
                                       return_sequences=True,
                                       return_state=True,
                                       recurrent_initializer='glorot_uniform')
        self.fc = tf.keras.layers.Dense(vocab_size)

        # Used for attention
        self.attention = BahdanauAttention(self.dec_units)

    def call(self, x, hidden, enc_output):
        # enc_output shape == (batch_size, max_length, hidden_size)
        context_vector, attention_weights = self.attention(hidden, enc_output)

        # x shape after passing through embedding == (batch_size, 1, embedding_dim)
        x = self.embedding(x)

        # x shape after concatenation == (batch_size, 1, embedding_dim + hidden_size)
        x = tf.concat([tf.expand_dims(context_vector, 1), x], axis=-1)

        # passing the concatenated vector to the GRU
        output, state = self.gru(x)

        # output shape == (batch_size * 1, hidden_size)
        output = tf.reshape(output, (-1, output.shape[2]))

        # output shape == (batch_size, vocab)
        x = self.fc(output)

        return x, state, attention_weights

    def initialize_hidden_state(self):
        return tf.zeros((self.batch_sz, self.dec_units))


# --- Minimal Demonstration ---

# Dummy data for demonstration
vocab_size = 1000
embedding_dim = 256
units = 512
batch_sz = 64
sequence_length = 10

# Create dummy input data
dummy_input_sequence = tf.random.uniform((batch_sz, sequence_length), maxval=vocab_size-1, dtype=tf.int32)
dummy_target_sequence = tf.random.uniform((batch_sz, sequence_length), maxval=vocab_size-1, dtype=tf.int32)

# Instantiate encoder
encoder = Encoder(vocab_size, embedding_dim, units, batch_sz)

# Initialize encoder hidden state
enc_hidden = encoder.initialize_hidden_state()

# Pass dummy input through encoder
enc_output, enc_hidden = encoder(dummy_input_sequence, enc_hidden)

# Instantiate decoder
decoder = Decoder(vocab_size, embedding_dim, units, batch_sz)

# Initialize decoder hidden state with encoder's last hidden state
dec_hidden = enc_hidden

# Use a dummy start token for the first decoder input
dec_input = tf.expand_dims([1] * batch_sz, 1) # Assuming 1 is a start token ID

# Pass through decoder for one step
predictions, dec_hidden, _ = decoder(dec_input, dec_hidden, enc_output)

print("--- Attention-based Encoder-Decoder Model Demo ---")
print(f"Encoder output shape: {enc_output.shape}") # (batch_sz, sequence_length, units)
print(f"Encoder hidden state shape: {enc_hidden.shape}") # (batch_sz, units)
print(f"Decoder predictions shape: {predictions.shape}") # (batch_sz, vocab_size)
print(f"Decoder hidden state shape: {dec_hidden.shape}") # (batch_sz, units)
print("Model components instantiated and shapes are consistent.")


--- Attention-based Encoder-Decoder Model Demo ---
Encoder output shape: (64, 10, 512)
Encoder hidden state shape: (64, 512)
Decoder predictions shape: (64, 1000)
Decoder hidden state shape: (64, 512)
Model components instantiated and shapes are consistent.


# question 9: generate a poem using a pre-trained gpt model.

In [3]:


!pip install transformers -q

from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="gpt2"
)

prompt = "roses are red, violets are blue,"

result = generator(
    prompt,
    max_length=40,
    num_return_sequences=1
)

print("prompt:")
print(prompt)

print("\ngenerated poem:")
print(result[0]["generated_text"])

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_length', 'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=40) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_

prompt:
roses are red, violets are blue,

generated poem:
roses are red, violets are blue, and the black and white are the same color.

The same way it can be done with water, the same way it can be done with any other kind of liquid.

The same way it can be done with any other kind of liquid.

The same way it can be done with any other kind of liquid.

The same way it can be done with any other kind of liquid.

The same way it can be done with any other kind of liquid.

The same way it can be done with any other kind of liquid.

The same way it can be done with any other kind of liquid.

The same way it can be done with any other kind of liquid.

The same way it can be done with any other kind of liquid.

The same way it can be done with any other kind of liquid.

The same way it can be done with any other kind of liquid.

The same way it can be done with any other kind of liquid.

The same way it can be done with any other kind of liquid.

The same way it can be done with any other k

# question 10: design a creative writing assistant using generative ai.

# answer:

# model:
# use gpt or llama for story and character generation.

# training data:
# books, novels, stories, and movie scripts.

# workflow:
# user enters a topic.
# the model generates story ideas and character descriptions.
# users can edit and regenerate content.

# bias mitigation:
# use filtered datasets.
# remove harmful content.
# apply human review.

# evaluation:
# check grammar.
# measure creativity.
# collect user feedback.

# challenges:
# copyright issues.
# biased outputs.
# incorrect information.
# high computing cost.
# maintaining originality.